In [57]:
import os
import numpy as np
import pandas as pd
import joblib

# Random Forest Models
from xgboost import XGBRegressor
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [58]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')

In [59]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [60]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet')).iloc[:, 0]
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet')).iloc[:, 0]
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet')).iloc[:, 0]

In [61]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [62]:
print(ID_test.head())
print(ID_test.columns)

  kabupaten_kota_asli  tahun
0       Kab. Balangan   2024
1       Kab. Balangan   2025
2     Kota Balikpapan   2024
3     Kota Balikpapan   2025
4         Kab. Banjar   2024
Index(['kabupaten_kota_asli', 'tahun'], dtype='object')


In [63]:
# Load metadata kolom (referensi)   
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))

In [64]:
# Verifikasi shape
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, ID_train: {ID_train.shape}")
print(f"X_val : {X_val.shape}, y_val : {y_val.shape}, ID_val : {ID_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}, ID_test : {ID_test.shape}")
print(f"\nTotal baris: {len(X_train) + len(X_val) + len(X_test)}")
print(f"Total fitur: {X_train.shape[1]} kolom")
print(f"Target : produktivitas padi (Qu/Ha)")
print(f"Range target train: {y_train.min():.2f} - {y_train.max():.2f}")

X_train: (278, 111), y_train: (278,), ID_train: (278, 2)
X_val : (56, 111), y_val : (56,), ID_val : (56, 2)
X_test : (111, 111), y_test : (111,), ID_test : (111, 2)

Total baris: 445
Total fitur: 111 kolom
Target : produktivitas padi (Qu/Ha)
Range target train: 15.78 - 54.46


In [65]:
# Setup Output Folder
MODEL_FOLDER = os.path.join(BASE_PATH, 'models')
RESULT_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

In [66]:
print("Mulai training XGBoost")

# XGBoost
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# training
xgb_model.fit(X_train, y_train)

print("Training selesai")

Mulai training XGBoost
Training selesai


In [67]:
# Prediksi
xgb_train_pred = xgb_model.predict(X_train)
xgb_val_pred = xgb_model.predict(X_val)
xgb_test_pred = xgb_model.predict(X_test)

In [68]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [69]:
# evaluasi train
xgb_train_mae, xgb_train_rmse, xgb_train_r2 = hitung_metrics(
    y_train,
    xgb_train_pred
)

In [70]:
# evaluasi validation
xgb_val_mae, xgb_val_rmse, xgb_val_r2 = hitung_metrics(
    y_val,
    xgb_val_pred
)

In [71]:
# evaluasi test
xgb_test_mae, xgb_test_rmse, xgb_test_r2 = hitung_metrics(
    y_test,
    xgb_test_pred
)

In [72]:
print("HASIL XGBoost")

print("\nTrain")
print(f"MAE  : {xgb_train_mae:.4f}")
print(f"RMSE : {xgb_train_rmse:.4f}")
print(f"R2   : {xgb_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {xgb_val_mae:.4f}")
print(f"RMSE : {xgb_val_rmse:.4f}")
print(f"R2   : {xgb_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {xgb_test_mae:.4f}")
print(f"RMSE : {xgb_test_rmse:.4f}")
print(f"R2   : {xgb_test_r2:.4f}")

HASIL XGBoost

Train
MAE  : 0.2048
RMSE : 0.2741
R2   : 0.9984

Validation
MAE  : 3.5430
RMSE : 4.5159
R2   : 0.5257

Test
MAE  : 3.7814
RMSE : 4.8418
R2   : 0.5625


In [73]:
# Simpan Model XGBoost
joblib.dump(
    xgb_model,
    os.path.join(BASE_PATH, 'models', 'xgboost.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [74]:
# Simpan metrics
metrics_xgb = pd.DataFrame({
    "model": ["XGBoost"],
    "train_mae": [xgb_train_mae],
    "train_rmse": [xgb_train_rmse],
    "train_r2": [xgb_train_r2],
    "val_mae": [xgb_val_mae],
    "val_rmse": [xgb_val_rmse],
    "val_r2": [xgb_val_r2],
    "test_mae": [xgb_test_mae],
    "test_rmse": [xgb_test_rmse],
    "test_r2": [xgb_test_r2]
})



os.makedirs("results", exist_ok=True)

metrics_xgb.to_csv(
    os.path.join(BASE_PATH, "results", "xgboost_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.
